In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("../../datasets/heart_final_cleaned.csv")
df = df.dropna().reset_index(drop=True)

X = df.drop("target", axis=1).values.astype(float)
y = df["target"].values.astype(int)

print("Dataset Shape:", df.shape)
print(df["target"].value_counts())

Dataset Shape: (302, 14)
target
1    164
0    138
Name: count, dtype: int64


In [3]:
def split_data(X, y, test_size=0.2, seed=42):

    rng = np.random.default_rng(seed)
    train_idx, test_idx = [], []

    for cls in np.unique(y):
        idx = np.where(y == cls)[0]
        rng.shuffle(idx)

        split = int(len(idx) * (1 - test_size))
        train_idx.extend(idx[:split])
        test_idx.extend(idx[split:])

    train_idx = np.array(train_idx)
    test_idx = np.array(test_idx)

    rng.shuffle(train_idx)
    rng.shuffle(test_idx)

    return X[train_idx], X[test_idx], y[train_idx], y[test_idx]


X_train, X_test, y_train, y_test = split_data(X, y)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

Training Shape: (241, 13)
Testing Shape: (61, 13)


In [4]:
mean = X_train.mean(axis=0)
std = X_train.std(axis=0)

std[std == 0] = 1

X_train = (X_train - mean) / std
X_test = (X_test - mean) / std

In [5]:
def euclidean_distance(x1, x2):

    return np.sqrt(np.sum((x1 - x2) ** 2))


distance = euclidean_distance(X_test[0], X_train[0])

print("Euclidean Distance:", distance)

Euclidean Distance: 4.422121739218623


In [6]:
def find_neighbors(X_train, query, k):

    distances = np.sqrt(np.sum((X_train - query) ** 2, axis=1))

    nearest_indices = np.argpartition(distances, k - 1)[:k]

    return nearest_indices, distances[nearest_indices]

In [8]:
def predict_knn(X_train, y_train, X_test, k=5):

    predictions = []

    for query in X_test:

        indices, distances = find_neighbors(X_train, query, k)

        neighbor_labels = y_train[indices]

        classes, counts = np.unique(
            neighbor_labels, return_counts=True
        )

        tied_classes = classes[counts == counts.max()]

        if len(tied_classes) == 1:
            prediction = tied_classes[0]

        else:
            nearest_order = np.argsort(distances)

            prediction = next(
                y_train[indices[i]]
                for i in nearest_order
                if y_train[indices[i]] in tied_classes
            )

        predictions.append(prediction)

    return np.array(predictions)

In [9]:
k = 5

y_pred = predict_knn(
    X_train, y_train, X_test, k
)

print("Actual Values:   ", y_test[:10])
print("Predicted Values:", y_pred[:10])

Actual Values:    [0 1 0 1 0 0 1 1 1 1]
Predicted Values: [1 1 0 0 0 0 1 1 1 1]


In [10]:
correct = sum(y_test == y_pred)

accuracy = (correct / len(y_test)) * 100

print("Total Predictions:", len(y_test))
print("Correct Predictions:", correct)
print("Incorrect Predictions:", len(y_test) - correct)
print("KNN Accuracy:", round(accuracy, 2), "%")

Total Predictions: 61
Correct Predictions: 54
Incorrect Predictions: 7
KNN Accuracy: 88.52 %


## Conclusion

In this task, a k-Nearest Neighbors classifier was implemented
from scratch without using scikit-learn.

Euclidean distance was manually calculated to measure the
similarity between samples. An efficient neighbor search
approach was used to identify the nearest training samples.

Majority voting was used for classification, and ties were
resolved by selecting the class of the nearest neighbor.

The model was evaluated using manually calculated accuracy.